# PlanForge
> AI business plan generator

*v0.3.0*

In [ ]:
# @title Setup
!pip install -q pysolvr-client
import sys
from google.colab import userdata, drive

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    from IPython.display import HTML, display
    display(HTML('<div style="color:#f87171;padding:12px;border:1px solid #f87171;border-radius:8px;font-family:Inter,sans-serif"><b>API key not found in Colab Secrets</b><ol><li>Click the key icon in the left sidebar</li><li>Add secret: PLANFORGE_API_KEY</li><li>Paste your API key as the value</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'PLANFORGE_API_KEY not found in Colab Secrets'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'scorecards'])
drive_mgr.ensure_folders()

if client.health_check():
    ui.success('Connected to PlanForge API', f'Drive: {drive_mgr.root}')
else:
    ui.error('Could not connect to API', 'Check your API key and try again')

In [ ]:
# @title Available Templates
result = client.call('GET', '/goals')
if result['ok']:
    goals = result['data']['goals']
    rows = [{'Template': g['name'], 'Audience': g['audience'], 'Topics': len(g['required_topics'])} for g in goals]
    ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch templates'))

In [ ]:
# @title Discovery Chat
# @markdown Have a conversation to build your business profile
TOPIC = 'business_overview'  # @param ["business_overview", "market", "financials", "team", "traction", "use_of_funds", "repayment", "innovation", "scalability", "job_creation", "founder_background", "assumptions", "risks", "impact", "project_plan", "coachability"]
MESSAGE = ''  # @param {type:"string"}

if not MESSAGE.strip():
    ui.error('Enter a message above', 'Describe your business idea or answer the question')
else:
    result = client.run_async('/chat', {'topic': TOPIC, 'message': MESSAGE})
    if result['ok']:
        data = result['data']
        ui.card('PlanForge', data.get('reply', ''), status='success')
        if data.get('locked'):
            ui.success(f'Topic locked!', f'{TOPIC} is complete. Move to the next topic.')
        if data.get('next_topic'):
            ui.card('Next', f"Suggested next topic: <b>{data['next_topic']}</b>")
    else:
        ui.error(result.get('error', 'Chat failed'), f'Status: {result.get("status")}')

In [ ]:
# @title View Profile
result = client.call('GET', '/profile')
if result['ok']:
    profile = result['data']
    topics = profile.get('topics', {})
    if not topics:
        ui.card('Profile', 'No topics discussed yet. Use Discovery Chat to build your profile.')
    else:
        rows = [{'Topic': k, 'Status': 'Locked' if v.get('locked') else 'In Progress', 'Turns': len(v.get('history', []))} for k, v in topics.items()]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch profile'))

In [ ]:
# @title Generate Plan
# @markdown Generate a business plan from your locked profile topics
TEMPLATE = 'scorecard'  # @param ["scorecard", "sba_loan", "innovator_visa", "seed_pitch", "innovate_uk", "bank_loan", "accelerator", "validate"]

result = client.run_async('/generate', {'goal_type': TEMPLATE})
if result['ok']:
    data = result['data']
    plan_id = data.get('plan_id', '')
    content = data.get('content', data.get('plan', ''))
    ui.card(f'Plan: {TEMPLATE}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{content}</pre>', status='success')
    if plan_id:
        # Save to Drive
        filename = f'{TEMPLATE}_{plan_id[:8]}.md'
        drive_mgr.save('plans', filename, content, meta={'plan_id': plan_id, 'template': TEMPLATE})
        ui.success(f'Saved to Drive', f'plans/{filename}')
else:
    ui.error(result.get('error', 'Generation failed'), f'Status: {result.get("status")}')

In [ ]:
# @title Plan History
result = client.call('GET', '/plans')
if result['ok']:
    plans = result['data'].get('plans', [])
    if not plans:
        ui.card('Plans', 'No plans generated yet. Use Generate Plan above.')
    else:
        rows = [{'ID': p['plan_id'][:8], 'Template': p.get('goal_type', ''), 'Created': p.get('created_at', '')[:10]} for p in plans]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch plans'))

In [ ]:
# @title Readiness Score
# @markdown Check if your profile is ready to generate a plan
GOAL_TYPE = 'scorecard'  # @param ["scorecard", "sba_loan", "innovator_visa", "seed_pitch", "innovate_uk", "bank_loan", "accelerator", "validate"]

result = client.call('POST', '/readiness', {'goal_type': GOAL_TYPE})
if result['ok']:
    data = result['data']
    emoji = {'ready': '\u2705', 'almost': '\u26a0\ufe0f', 'not_ready': '\u274c'}.get(data['verdict'], '')
    ui.card(f"{emoji} Readiness: {data['score']}/100", f"<b>Verdict:</b> {data['verdict']}<br><b>Next:</b> {data['recommended_next']}", status='success' if data['verdict'] == 'ready' else 'warning')
    if data.get('gaps'):
        rows = [{'Topic': g['topic'], 'Action': g['action'], 'Priority': g['priority']} for g in data['gaps']]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Readiness check failed'))


In [ ]:
# @title Export Plan
# @markdown Reshape your plan into a different format
FORMAT = 'executive_summary'  # @param ["executive_summary", "pitch_deck", "one_pager", "financial_summary", "full_pdf"]

# Load latest plan from Drive
plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    result = client.run_async('/export', {'content': content, 'format': FORMAT})
    if result['ok']:
        data = result['data']
        ui.card(f'Export: {FORMAT}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
        filename = f'{FORMAT}_{plan_files[-1].replace(".md", "")}.md'
        drive_mgr.save('plans', filename, data['content'], meta={'format': FORMAT})
        ui.success('Saved to Drive', f'plans/{filename}')
    else:
        ui.error(result.get('error', 'Export failed'), f'Status: {result.get("status")}')

In [ ]:
# @title Refine Section
# @markdown Regenerate a specific section with optional feedback
SECTION = ''  # @param {type:"string"}
FEEDBACK = ''  # @param {type:"string"}
ADDITIONAL_CONTEXT = ''  # @param {type:"string"}

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
elif not SECTION.strip():
    ui.error('Enter a section name', 'e.g. Market Analysis, Financial Projections')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    body = {'content': content, 'section': SECTION}
    if FEEDBACK.strip():
        body['feedback'] = FEEDBACK
    if ADDITIONAL_CONTEXT.strip():
        body['additional_context'] = ADDITIONAL_CONTEXT
    result = client.run_async('/refine', body)
    if result['ok']:
        data = result['data']
        ui.card(f'Refined: {data["section"]}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
    else:
        ui.error(result.get('error', 'Refinement failed'), f'Status: {result.get("status")}')

In [ ]:
# @title Account
result = client.call('GET', '/account')
if result['ok']:
    d = result['data']
    ui.card('Account', (
        f"<b>Plan:</b> {d.get('plan', 'free')}<br>"
        f"<b>Spend:</b> ${d.get('monthly_spend_usd', 0):.2f} / ${d.get('monthly_limit_usd', 0):.2f}<br>"
        f"<b>Top-up:</b> ${d.get('topup_balance_usd', 0):.2f}"
    ), status='success')
else:
    ui.error(result.get('error', 'Could not fetch account'))

In [ ]:
# @title Usage
result = client.get_usage()
if result['ok']:
    data = result['data']
    ui.usage_bar(data.get('current_month_spend_usd', 0), data.get('monthly_limit_usd', 1))
    if data.get('recent'):
        ui.table(data['recent'])
else:
    ui.error(result.get('error', 'Could not fetch usage'), 'Check your API key')

In [ ]:
# @title Help
from IPython.display import HTML, display
display(HTML(
    '<details style="margin:8px 0;font-family:Inter,system-ui,sans-serif;color:#f1f5f9">'
    '<summary style="cursor:pointer;font-weight:500">Quick Start</summary>'
    '<ol style="color:#94a3b8;font-size:13px;padding-left:20px">'
    '<li>Run Setup with your API key</li>'
    '<li>Use Discovery Chat to describe your business</li>'
    '<li>Generate plans for different audiences</li>'
    '<li>Plans saved to Google Drive automatically</li>'
    '</ol></details>'
    '<details style="margin:8px 0;font-family:Inter,system-ui,sans-serif;color:#f1f5f9">'
    '<summary style="cursor:pointer;font-weight:500">FAQ</summary>'
    '<p style="color:#94a3b8;font-size:13px"><b>Files?</b> Drive > pysolvr > planforge</p>'
    '<p style="color:#94a3b8;font-size:13px"><b>Help?</b> support@pysolvr.com</p>'
    '</details>'
))